Cell 1: Handles Imports

In [56]:
import pandas
from pathlib import Path

Cell 2: Finds raw data from data directory

In [57]:
raw_dir = Path("../data/raw")
raw_files = sorted(list(raw_dir.glob("*.csv")))
## print(f"Found {len(raw_files)} raw files: {raw_files}") check


Cell 3: Add all the seasons into one list and add a label for each respective season

In [58]:
all_seasons = []
for file in raw_files:
    print(f"Processing {file}...")
    season_df = pandas.read_csv(file)
    season_df = season_df.dropna(subset=["HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"])
    season_year = int(file.stem.split("_")[2])
    season_df["Season"] = season_year
    ## print (f"Season df shape: {season_df.shape}, {season_df.Season.unique()}")
    all_seasons.append(season_df)
    print(f"Finished processing {file}.")
    
## Season is an integer that represents the starting year of the season. For example, the 2014-2015 season is represented as 2014. The last season in the dataset is 2025-2026, which is represented as 2025.


Processing ../data/raw/premier_league_2014_2015.csv...
Finished processing ../data/raw/premier_league_2014_2015.csv.
Processing ../data/raw/premier_league_2015_2016.csv...
Finished processing ../data/raw/premier_league_2015_2016.csv.
Processing ../data/raw/premier_league_2016_2017.csv...
Finished processing ../data/raw/premier_league_2016_2017.csv.
Processing ../data/raw/premier_league_2017_2018.csv...
Finished processing ../data/raw/premier_league_2017_2018.csv.
Processing ../data/raw/premier_league_2018_2019.csv...
Finished processing ../data/raw/premier_league_2018_2019.csv.
Processing ../data/raw/premier_league_2019_2020.csv...
Finished processing ../data/raw/premier_league_2019_2020.csv.
Processing ../data/raw/premier_league_2020_2021.csv...
Finished processing ../data/raw/premier_league_2020_2021.csv.
Processing ../data/raw/premier_league_2021_2022.csv...
Finished processing ../data/raw/premier_league_2021_2022.csv.
Processing ../data/raw/premier_league_2022_2023.csv...
Finished 

/var/folders/p1/sn4dbmvj6573lvrjfgfkyj000000gn/T/ipykernel_74363/171165241.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  season_df["Season"] = season_year


Cell 4: Combine all the single season data frames into 1 data frame

In [59]:
df = pandas.concat(all_seasons, ignore_index=True)
print(f"Combined dataframe shape: {df.shape}, Seasons: {df.Season.unique()}")

Combined dataframe shape: (4560, 184), Seasons: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]


Cell 5: Checking the data

In [60]:
print(df["Season"].value_counts())
df[["Season", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Date", "HS", "AS", "HST", "AST"]].isna().sum()

Season
2014    380
2015    380
2016    380
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    380
2024    380
2025    380
Name: count, dtype: int64


Season      0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
Date        0
HS          0
AS          0
HST         0
AST         0
dtype: int64

Cell 6: Finding columns that are present in every match

In [61]:
common_columns = set(all_seasons[0].columns)
for i in range(1, len(all_seasons)):
    common_columns = common_columns.intersection(set(all_seasons[i].columns))
common_columns = sorted(common_columns)
print(f"Common columns across all seasons: {common_columns}")

Common columns across all seasons: ['AC', 'AF', 'AR', 'AS', 'AST', 'AY', 'AwayTeam', 'B365A', 'B365D', 'B365H', 'BWA', 'BWD', 'BWH', 'Date', 'Div', 'FTAG', 'FTHG', 'FTR', 'HC', 'HF', 'HR', 'HS', 'HST', 'HTAG', 'HTHG', 'HTR', 'HY', 'HomeTeam', 'PSA', 'PSCA', 'PSCD', 'PSCH', 'PSD', 'PSH', 'Referee', 'Season']


Cell 7: Finding bookmaker odds present in every match

In [62]:
print(df[["PSH", "PSA", "PSCH", "PSCD", "PSCA", "PSD"]].isna().sum())
mask = df["PSH"].isna()
missing_psh = df[mask]
print(missing_psh["Season"].value_counts())
print(f"Number of rows with missing PSH: {missing_psh.shape[0]}")
print()
print(df[["B365H", "B365D", "B365A"]].isna().sum())

PSH     170
PSA     170
PSCH    170
PSCD    170
PSCA    170
PSD     170
dtype: int64
Season
2025    170
Name: count, dtype: int64
Number of rows with missing PSH: 170

B365H    0
B365D    0
B365A    0
dtype: int64


Ideally wanted to use pinnacle odds, but they restricted access to odds in 2025, leaving gaps in the data for 170 of the 2025/26 matches. Bet 365 bookmaker odds are present for every match, so we will continue with those

Cell 8: Keep certain columns

In [63]:
keep_cols = ["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HS", "AS", "HST", "AST", "B365H", "B365D", "B365A", "HC", "AC", "HY", "AY", "HR", "AR"]
matches = df[keep_cols].copy()
print(f"Final matches dataframe shape: {matches.shape}, Seasons: {matches.Season.unique()}")


Final matches dataframe shape: (4560, 20), Seasons: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]


Cell 9: Parsing Dates

In [64]:
#Check that dates have same format across all seasons
#Should check if dates are kept day first or month first

#Based on prints -> Day First format
#One thing, early dates are 2 digit in the year, later are 4 digits
matches["Date"] = pandas.to_datetime(matches["Date"], dayfirst=True, format="mixed")
print(matches["Date"].isna().sum()) #Check if any dates are missing after conversion
print()
print(matches.groupby("Season")["Date"].agg(["min", "max"]))
print()
print(matches["Date"].head())
print()
print(matches["Date"].tail())


0

              min        max
Season                      
2014   2014-08-16 2015-05-24
2015   2015-08-08 2016-05-17
2016   2016-08-13 2017-05-21
2017   2017-08-11 2018-05-13
2018   2018-08-10 2019-05-12
2019   2019-08-09 2020-07-26
2020   2020-09-12 2021-05-23
2021   2021-08-13 2022-05-22
2022   2022-08-05 2023-05-28
2023   2023-08-11 2024-05-19
2024   2024-08-16 2025-05-25
2025   2025-08-15 2026-05-24

0   2014-08-16
1   2014-08-16
2   2014-08-16
3   2014-08-16
4   2014-08-16
Name: Date, dtype: datetime64[us]

4555   2026-05-24
4556   2026-05-24
4557   2026-05-24
4558   2026-05-24
4559   2026-05-24
Name: Date, dtype: datetime64[us]


Cell 10: Team name spelling errors

In [65]:
##Check that all teams are consistent across seasons
##Check that home and awat teasm match exactly

print(sorted(matches["HomeTeam"].unique()))
print(sorted(matches["AwayTeam"].unique()))

['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']


Cell 11: Sort and Save

In [66]:
matches = matches.sort_values("Date").reset_index(drop=True)
matches.to_csv("../data/processed/matches.csv", index=False)
check = pandas.read_csv("../data/processed/matches.csv")
print(f"Check dataframe shape: {check.shape}, Seasons: {check.Season.unique()}, Dates: {check['Date'].dtype}")


Check dataframe shape: (4560, 20), Seasons: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025], Dates: str
